# 06 - Evaluation comparison

All numbers are read from saved results on the frozen 104-case `golden_v2_india` set. Baseline and v1-v5 are historical Claude measurements; v6 uses Nebius DeepSeek. A complete `v7_vector` result, when present, is labelled separately: forced vector retrieval, stage source filters, GLM MAIN, DeepSeek FAST, lessons disabled. No paid or network calls run from this notebook. Code and judge-error handling live in `evals/compare.py` and `evals/judge.py`.

In [1]:
from evals import compare, report

versions = report.versions()
table = compare.comparison_table(versions=versions)
print(compare.format_table(table))

version                 esc             refuse              route               cite              faith              p95 s            mean Rs             p95 Rs
baseline             100.0%            97.1% X            84.6% X              78.1%            89.3% X             8.57 X              1.305              1.502
v1           100.0% (+0.0%)    99.0% (+1.9%) X    79.8% (-4.8%) X      82.2% (+4.1%)    85.7% (-3.6%) X       7.58 (-0.99)     1.007 (-0.298)     1.513 (+0.011)
v2           100.0% (+0.0%)    99.0% (+0.0%) X    84.6% (+4.8%) X      80.8% (-1.4%)    79.4% (-6.3%) X     8.31 (+0.73) X      0.917 (-0.09)     1.541 (+0.028)
v3           100.0% (+0.0%)    99.0% (+0.0%) X    82.7% (-1.9%) X      78.1% (-2.7%)    76.1% (-3.3%) X    10.55 (+2.24) X     0.951 (+0.034)     1.555 (+0.014)
v4           100.0% (+0.0%)    99.0% (+0.0%) X    81.7% (-1.0%) X     67.1% (-11.0%)     94.1% (+18.0%)    16.91 (+6.36) X      1.971 (+1.02)    4.625 (+3.07) X
v5           100.0% (+0.0%)     10

## Latest completed measurement

Pass bars remain unchanged. Judge errors are excluded from the faithfulness denominator and named explicitly. The app fallback and lessons-enabled modes are separate from these measurements.

In [2]:
latest = versions[-1]
print("Latest completed base:", latest)
print("Missed bars:", [k for k, ok in table[-1]["passed"].items() if ok is False])
print("Judge summary:", compare.judge_summary(latest))
print("Judge-error IDs:", [r["case_id"] for r in compare.load_judge(latest) if r["faithful"] is None])

Latest completed base: v7_vector_safe
Missed bars: ['faithfulness', 'p95_latency_s']
Judge summary: {'faithfulness': 0.8636, 'mean_claim_support': 0.9485, 'n_judged': 66, 'n_judge_errors': 2, 'pass_bar': '>= 0.9', 'judge': 'Qwen/Qwen3-235B-A22B-Instruct-2507', 'judge_version': 'judge-v1'}
Judge-error IDs: ['B13', 'F09']


## Faithfulness regressions

A regression means the same case was judged faithful before and unfaithful after. Provider and safety-rule changes prevent assigning every delta to a single lever.

In [3]:
pairs = [("v4", "v5"), ("v5", "v6")]
if latest != "v6":
    pairs.append(("v6", latest))
for before, after in pairs:
    rows = compare.faithfulness_regressions(before, after)
    print(before, "to", after, len(rows), "regressions")
    for row in rows:
        print(row["case_id"], row["agent"], row["unsupported"][:220])

v4 to v5 9 regressions
B06 care If you are 32 weeks pregnant, you should know these danger signs.
B13 care Bring your baby's vaccination card so they can see what has been given.
E02 care A mild headache with no other symptoms is very common in pregnancy.
F05 ready Photo ID and insurance details are needed
F08 money Please upload the policy document, or ask your HR team, insurer or TPA directly. | Good things to ask: is maternity covered, is there a waiting period, what is the sum insured, and is the baby covered from birth. | So it
F14 care keep the napkin area, skin folds and cord area properly dried with a soft cloth when bathing (MOHFW-ANC, p. 68) | keep the cord and umbilicus dry without applying anything on it (MOHFW-ANC, p. 64) | Watch breastfeeding a
H08 nourish at 18 weeks you are in the second trimester | you avoid root vegetables
H16 ready Go with her to her 34-week visit if you can.
M09 care Cutting out rice completely fixes gestational diabetes.
v5 to v6 10 regressions
B14

## Current failure modes

A case can appear in several categories. Citation match is diagnostic and has no separate pass bar.

In [4]:
for item in compare.top_failure_modes(latest):
    print(item["mode"], item["count"], ", ".join(item["case_ids"]))
print("Citation-match misses:", len(compare.citation_misses(latest)))

wrong source family 22 B09, B11, B13, B14, F01, F03, F06, F08, F14, F15, F21, H06, H09, H10, H18, M02, M03, M10, M15, M18, M20, M23
unsupported claim 9 B14, F01, F05, F08, F10, H06, H15, M06, M09
routing boundary 6 E03, H16, M02, M03, M16, M23
Citation-match misses: 28


## Run status

The resumable budget runner has a read-only local status command: `uv run python -m evals.budgeted_run --status`. Only a complete 104-case checkpoint plus eligible judge verdicts is exported as a measurement. The report records failed bars honestly and human-vs-judge calibration remains pending actual human labels. Strict-reranked v7 has no full result; the subset model bake-off is not a full-set score.